# SAHA: CosMx 6K Spatial Treatment Validation (Fig 6 / Supp Fig 10)

CosMx 6K spatial analysis of IBD treatment cohort (SAHA).
Compares Treg pseudotime state and fibroblast neighborhood across three treatment groups:
pre-treatment (`SAHA_pre_trt`), post-treatment remission (`SAHA_post_trt_remission`),
and post-treatment non-remission (`SAHA_post_trt_nonremission`).

**Run `01_saha_anno.R` first** to generate AUCell annotation and Treg label transfer outputs.


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scanpy.external as sce

from scipy.sparse import csr_matrix

In [ ]:
saha_6k=sc.read_h5ad('/path/to/saha/SAHA_IBD_RNA.h5ad')

In [ ]:
saha_6k.obs['SAHA_Sample'].value_counts()

In [ ]:
# treatment samples a2_1 is pre-trt, a2_2 is post trt same sample in remission, a1_2 is post trt non-remission
saha_6k=saha_6k[saha_6k.obs['SAHA_Sample'].isin(['SAHA_pre_trt','SAHA_post_trt_remission','SAHA_post_trt_nonremission'])]

In [ ]:
adata = saha_6k.copy()

In [ ]:
adata.var["neg_probe"]     = adata.var_names.str.contains("Negative")
adata.var["control_probe"] = adata.var_names.str.contains("SystemControl")
adata = adata[:, ~(adata.var["neg_probe"] | adata.var["control_probe"])].copy()
print(f"After probe removal: {adata.n_vars} genes")

In [ ]:
adata.X = adata.layers['raw']

In [ ]:
if not isinstance(adata.X, csr_matrix):
    adata.X = csr_matrix(adata.X)

sc.pp.calculate_qc_metrics(adata, inplace=True, log1p=False)

plot_keys = ["n_genes_by_counts", "total_counts"]
if "Area" in adata.obs.columns:
    plot_keys.append("Area")

sc.pl.violin(adata, plot_keys, jitter=0.4, multi_panel=True, show=True)


In [ ]:
min_counts = 200
max_counts = 5000
min_genes  = 100
max_genes  = 3000
max_area   = 40000

In [ ]:
n_before = adata.n_obs
sc.pp.filter_cells(adata, min_counts=min_counts)
sc.pp.filter_cells(adata, max_counts=max_counts)
sc.pp.filter_cells(adata, min_genes=min_genes)
sc.pp.filter_cells(adata, max_genes=max_genes)
if "Area" in adata.obs.columns and max_area:
    adata = adata[adata.obs["Area"] <= max_area].copy()
print(f"QC filtering: {n_before} → {adata.n_obs} cells (removed {n_before - adata.n_obs})")


In [ ]:
sc.pl.violin(adata, plot_keys, jitter=0.4, multi_panel=True, show=True)

In [ ]:
sc.pp.normalize_total(adata, inplace=True)

In [ ]:
outdir     = '/path/to/saha/'
csv_dir = os.path.join(outdir, "csv")
os.makedirs(csv_dir, exist_ok=True)

In [ ]:
df = pd.DataFrame(
    adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X,
    columns=adata.var_names,
    index=adata.obs_names)

In [ ]:
csv_path = os.path.join(csv_dir, f"6k_trt_norm_counts.csv")
df.to_csv(csv_path)
print(f"Saved: {csv_path}")

In [ ]:
# anno using aucell
anno =pd.read_csv('/path/to/saha/anno/6k_trt_norm_aucell.csv')

In [ ]:
mapper= pd.read_csv('/path/to/scrna/cd/cell_type_category_map.csv')

In [ ]:
# Step 1: Clean the names
mapper['cell_type_aucell'] = mapper['cell type'].str.replace(r'[+\-\/()]', ' ', regex=True)
mapper['cell_type_aucell'] = mapper['cell_type_aucell'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Step 2: Disambiguate duplicates by appending " 1", " 2", etc.
mapper['cell_type_aucell'] = (
    mapper.groupby('cell_type_aucell').cumcount().astype(str).replace('0', '', regex=False)
    .radd(mapper['cell_type_aucell'] + ' ').str.strip()
)

In [ ]:
anno['label_clean'] = anno['label'].str.rstrip()

In [ ]:
anno_map=anno.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')

In [ ]:
adata.obs['cell_type'] = anno_map['cell type'].tolist()
adata.obs['cell_type_short'] = anno_map['cell type short'].tolist()
adata.obs['cell_category'] = anno_map['cell category'].tolist()
adata.obs['cell_type_general'] = anno_map['cell type general'].tolist()

In [ ]:
tregs = adata[adata.obs['cell_type']=='Treg']

In [ ]:
tregs.obs['SAHA_Sample'].value_counts()

In [ ]:
sc.pp.highly_variable_genes(
    tregs, n_top_genes=3000, flavor='seurat_v3',layer='raw'
)

In [ ]:
sc.tl.pca(tregs, n_comps=30, use_highly_variable=True, )
# Harmony integrates on PCA embeddings; writes to X_pca_harmony
sce.pp.harmony_integrate(tregs, key='SAHA_Sample')  # or 'batch'

sc.pp.neighbors(tregs, use_rep='X_pca_harmony', n_neighbors=20, n_pcs=30)
sc.tl.umap(tregs)


In [ ]:
panels = {
 'Activated_tissue': ['BATF','PRDM1','MAF','TNFRSF18','TNFRSF4'],
 'Tfr': ['BCL6','CXCR5','PDCD1','ICOS'],
 'Th17_like': ['RORC','CCR6','IL17A','IL17F'],
 'IFN_response': ['ISG15','IFI6','MX1','IFIT3'],
 'Cell_cycle': ['MKI67','TOP2A','STMN1'],
 'Treg_core':['FOXP3','IL2RA','CTLA4','TIGIT','ENTPD1','LRRC32','IL10']
}
for name, genes in panels.items():
    sc.tl.score_genes(tregs, [g for g in genes if g in tregs.var_names], score_name=name)

In [ ]:
for name in panels.keys():
    sc.pl.umap(
        tregs,
        color=name,
        cmap="viridis",     # or 'Reds', 'coolwarm', etc.
        vmin='p1', vmax='p99',  # clip extreme values for nicer contrast
        frameon=False,size =20
    )


In [ ]:
raw_counts = pd.DataFrame(tregs.layers['raw'].toarray())
raw_counts.columns = tregs.var_names.tolist()
raw_counts.index = tregs.obs['cell'].tolist()

In [ ]:
raw_counts.to_csv('/path/to/saha/csv/6k_trt_treg_raw_counts.csv')

In [ ]:
# label transfer in r
treg_label = pd.read_csv('/path/to/saha/csv/6k_trt_reg_pt_transfer_labels.csv')

In [ ]:
treg_label['bin']=['Late' if i == 4 else 'Early' for i in treg_label['predicted.id'].tolist()]

In [ ]:
tregs.obs['Treg_bin']=['Treg_'+str(i) for i in treg_label['bin'].tolist()]

In [ ]:
# Map Treg_bin from treg adata back to full adata
treg_bin_map = tregs.obs['Treg_bin'].to_dict()  # index → Treg_bin

def assign_cell_type_treg(row):
    if row['cell_type_short'] == 'Treg':
        return treg_bin_map.get(row.name, 'Treg')  # fallback to 'Treg' if not found
    return row['cell_type_short']

adata.obs['cell_type_treg'] = adata.obs.apply(assign_cell_type_treg, axis=1)

# Verify
print(adata.obs['cell_type_treg'].value_counts())
print(f"\nTreg bins assigned: {adata.obs[adata.obs['cell_type_short'] == 'Treg']['cell_type_treg'].value_counts()}")

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))

data = pd.DataFrame({
    'Sample': ['SAHA_post_trt_nonremission\n(Pre)', 'SAHA_pre_trt\n(Post)'],
    'Early': [115, 160],
    'Late':  [26, 258]
})

data_pct = data.copy()
total = data[['Early','Late']].sum(axis=1)
data_pct['Early'] = data['Early'] / total * 100
data_pct['Late']  = data['Late']  / total * 100

data_pct.plot(
    x='Sample', kind='bar', stacked=True,
    color=['#378ADD', '#D85A30'],
    width=0.3, ax=ax, edgecolor='none'
)

ax.set_xlabel('')
ax.set_ylabel('Treg proportion (%)')
ax.set_title('Treg state pre→post treatment')
ax.set_xticklabels(['Pre-treatment\n(SAHA_post_trt_nonremission)', 'Post-treatment\n(SAHA_pre_trt)'],
                    rotation=0)
ax.legend(['Early Treg', 'Late Treg'], frameon=False, fontsize=9)
ax.set_ylim(0, 100)

# Annotate percentages
for i, (early_pct, late_pct) in enumerate(zip(data_pct['Early'], data_pct['Late'])):
    ax.text(i, early_pct/2, f'{early_pct:.0f}%', ha='center', va='center',
            color='white', fontsize=10, fontweight='bold')
    ax.text(i, early_pct + late_pct/2, f'{late_pct:.0f}%', ha='center', va='center',
            color='white', fontsize=10, fontweight='bold')

sns.despine()
plt.tight_layout()
plt.savefig('treg_prepost_saha.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import scipy.stats as stats
from statsmodels.nonparametric.smoothers_lowess import lowess

fig, ax = plt.subplots(figsize=(3.5,3.5))

sample = 'SAHA_pre_trt'
sub = adata[adata.obs['SAHA_Sample'] == sample].copy()

fov_stats = (
    sub.obs.groupby('fov')
    .apply(lambda x: pd.Series({
        'n_late_treg': (x['cell_type_treg'] == 'Treg_Late').sum(),
        'n_ogn_fibro': (x['cell_type_short'] == 'OGN+RSPO3+ Fib').sum(),
    }))
    .reset_index()
)

# Only FOVs with both cell types present
fov_nonzero = fov_stats[
    (fov_stats['n_ogn_fibro'] > 0) &
    (fov_stats['n_late_treg'] > 0)
].copy()

r, p = spearmanr(fov_nonzero['n_ogn_fibro'], fov_nonzero['n_late_treg'])

# Linear fit
x = fov_nonzero['n_ogn_fibro'].values
y = fov_nonzero['n_late_treg'].values
m, b = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
y_pred = m * x_line + b

# 95% confidence interval
n = len(x)
se = np.sqrt(np.sum((y - (m*x + b))**2) / (n - 2))
ci = 1.96 * se * np.sqrt(1/n + (x_line - x.mean())**2 /
                          np.sum((x - x.mean())**2))

# Plot
ax.scatter(x, y, color='#378ADD', s=40,
           edgecolor='white', linewidth=0.8,
           alpha=0.8, zorder=3)

ax.plot(x_line, y_pred, 'k--', lw=1.5, alpha=0.8, label='Linear fit')

ax.fill_between(x_line,
                y_pred - ci,
                y_pred + ci,
                alpha=0.15, color='gray', label='95% CI')

ax.set_xlabel('OGN⁺RSPO3⁺ Fibroblasts per FOV', fontsize=9)
ax.set_ylabel('Late Tregs per FOV', fontsize=9)
ax.set_title(f'Treg-Fibroblast Co-Localizaton \n'
             f'Spearman r={r:.3f}, p<0.001',
             fontsize=9)
ax.legend(frameon=False, fontsize=8)
ax.tick_params(labelsize=8)
ax.legend(frameon=False, fontsize=8, loc='lower right')
sns.despine()
plt.tight_layout()
plt.savefig('fov_coloc_remission_ci.pdf', dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
# 1. Remove zeros — only FOVs with both cell types present
fov_nonzero = fov_stats[(fov_stats['n_ogn_fibro'] > 0) & 
                         (fov_stats['n_late_treg'] > 0)]
r_nz, p_nz = spearmanr(fov_nonzero['n_ogn_fibro'], fov_nonzero['n_late_treg'])
print(f"With zeros: r={r:.3f}, n={len(fov_stats)}")
print(f"Without zeros: r={r_nz:.3f}, n={len(fov_nonzero)}")

# 2. Remove outlier (top right point)
fov_no_outlier = fov_stats[fov_stats['n_ogn_fibro'] < 80]
r_no, p_no = spearmanr(fov_no_outlier['n_ogn_fibro'], fov_no_outlier['n_late_treg'])
print(f"Without outlier: r={r_no:.3f}, p={p_no:.3f}, n={len(fov_no_outlier)}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(20, 20))

# Define a color palette for cell type categories
cell_types = adata.obs['cell_category'].unique()
palette = dict(zip(cell_types, 
                   sns.color_palette('tab20', len(cell_types))))

for ax, sample, title in zip(axes, samples, titles):
    sub = adata[adata.obs['SAHA_Sample'] == sample].copy()
    coords = sub.obsm['spatial']
    
    # Plot each cell type
    for ct in sub.obs['cell_category'].unique():
        mask = sub.obs['cell_category'] == ct
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=[palette[ct]], s=3, alpha=0.5,
                   label=ct, rasterized=True)
    
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('X (px)')
    ax.set_ylabel('Y (px)')
    ax.set_aspect('equal')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left',
              fontsize=6, markerscale=5, frameon=False,
              ncol=2)
    sns.despine(ax=ax)

plt.suptitle('Cell type spatial distribution', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('celltype_spatial.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
eps_px = 100 / 0.18
db = DBSCAN(eps=eps_px, min_samples=30).fit(bcell_coords)
labels = db.labels_

n_blobs = len(set(labels)) - (1 if -1 in labels else 0)
print(f"Blobs found: {n_blobs}")
for i in range(n_blobs):
    print(f"  TLS {i+1}: {(labels == i).sum()} cells")
print(f"  Noise: {(labels == -1).sum()} cells")

blob_centroids = np.array([
    bcell_coords[labels == i].mean(axis=0)
    for i in range(n_blobs)
])

# Recompute distances with new blobs
blob_tree = KDTree(blob_centroids)
blob_dists, _ = blob_tree.query(coords[treg_mask])
blob_dists_um = blob_dists * 0.18
proximity = 1 / (blob_dists_um + 1)

tfr_vals_arr = sub.obs.loc[treg_mask, 'tfr_score'].values
r, p = spearmanr(tfr_vals_arr, proximity)

close = blob_dists_um < 300
far   = ~close
_, p_mw = mannwhitneyu(tfr_vals_arr[close], tfr_vals_arr[far], alternative='greater')

print(f"\nTfr vs blob proximity: r={r:.3f}, p={p:.3f}")
print(f"Near (<300µm): {tfr_vals_arr[close].mean():.4f} (n={close.sum()})")
print(f"Far: {tfr_vals_arr[far].mean():.4f} (n={far.sum()}), p={p_mw:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(13,8))

sample = 'SAHA_pre_trt'
title  = ''

sub = adata[adata.obs['SAHA_Sample'] == sample].copy()
coords = sub.obsm['spatial']

epi_mask   = sub.obs['cell_category'] == 'Epithelial'
bcell_mask = sub.obs['cell_type_short'] == 'B cell'
treg_mask  = sub.obs['cell_type_short'] == 'Treg'
fibro_mask = sub.obs['cell_type_short'] == 'OGN+RSPO3+ Fib'
other_mask = ~(epi_mask | bcell_mask | treg_mask | fibro_mask)

# Background
ax.scatter(coords[other_mask, 0], coords[other_mask, 1],
           c='lightgray', s=0.8, alpha=0.5, rasterized=True)

# Epithelial
if epi_mask.sum() > 0:
    ax.scatter(coords[epi_mask, 0], coords[epi_mask, 1],
               c='#83ABDE', s=1, alpha=0.5,
               zorder=2, rasterized=True,
               label=f'Epithelial (n={epi_mask.sum()})')

# B cells
if bcell_mask.sum() > 0:
    ax.scatter(coords[bcell_mask, 0], coords[bcell_mask, 1],
               c='#2CA02C', s=3, alpha=0.7,
               zorder=3, rasterized=True,
               label=f'B cell (n={bcell_mask.sum()})')

# Tregs — smaller dots
if treg_mask.sum() > 0:
    tfr_vals = sub.obs.loc[treg_mask, 'tfr_score']
    sc2 = ax.scatter(coords[treg_mask, 0], coords[treg_mask, 1],
                     c=tfr_vals, cmap='Oranges',
                     s=4, alpha=0.9, zorder=5,
                     vmin=tfr_vals.quantile(0.05),
                     vmax=tfr_vals.quantile(0.95))
    plt.colorbar(sc2, ax=ax, label='Tfr score (Tregs)',
                 shrink=0.3, pad=0.01)
# OGN+RSPO3+ fibroblasts — purple
if fibro_mask.sum() > 0:
    ax.scatter(coords[fibro_mask, 0], coords[fibro_mask, 1],
               c='#9467BD', s=1.5, alpha=0.9,
               zorder=4, rasterized=True,
               label=f'OGN⁺RSPO3⁺ Fib (n={fibro_mask.sum()})')
    
# B cell blob centroids
for i, centroid in enumerate(blob_centroids):
    ax.scatter(centroid[0], centroid[1],
               c='black', s=80, marker='x',
               zorder=6, linewidths=1.5)
    ax.annotate(f'TLS {i+1}', centroid,
                fontsize=13, color='black',
                fontweight='bold',
                xytext=(5, 5), textcoords='offset points',
                ha='left', va='bottom',
                zorder=10)

# Stats
stats_str = (
    f'Tfr Score near (<300µm) = {tfr_vals_arr[close].mean():.3f} '
    f'vs far = {tfr_vals_arr[far].mean():.3f} to TLS, p={p_mw:.3f}  '
)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='lightgray',
           markersize=8, label='Other cells'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#83ABDE',
           markersize=8, label=f'Epithelial (n={epi_mask.sum()})'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#2CA02C',
           markersize=8, label=f'B cell (n={bcell_mask.sum()})'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#9467BD',
           markersize=8, label=f'OGN⁺RSPO3⁺ Fib (n={fibro_mask.sum()})'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='orange',
           markersize=8, label=f'Treg Tfr score (n={treg_mask.sum()})'),
    Line2D([0],[0], marker='x', color='black',
           markersize=8, label='TLS centroid'),
]
ax.legend(handles=legend_elements, frameon=False, fontsize=10,
          bbox_to_anchor=(0.99, 1.15), loc='upper left')

ax.set_title(f'{title}\n{stats_str}', fontsize=11)
# Convert axis ticks to microns
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x * 0.18:.0f}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x * 0.18:.0f}'))
ax.set_xlabel('X (um)')
ax.set_ylabel('Y (um)')
ax.set_aspect('equal')
ax.set_xlim(left=5000,right=37000)
ax.set_ylim(bottom=-4000,top=10000)
sns.despine()
plt.tight_layout()
#plt.savefig('tfr_bcell_remission_final.pdf',
            #dpi=600,           # increase from 300 to 600 for sharpness
            #bbox_inches='tight',
            #format='pdf')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))

for status, color in [('Treg_Early', '#378ADD'), ('Treg_Late', '#D85A30')]:
    vals = treg_rem.obs[treg_rem.obs['cell_type_treg'] == status]['tfr_score']
    sns.kdeplot(vals, ax=ax, color=color, fill=True,
                alpha=0.4, linewidth=2, label=status)

ax.axvline(g1.median(), color='#378ADD', linestyle='--', lw=1.5, alpha=0.8)
ax.axvline(g2.median(), color='#D85A30', linestyle='--', lw=1.5, alpha=0.8)

ax.set_xlabel('Tfr score')
ax.set_ylabel('Density')
ax.set_title(f'Tfr Score by Treg Pseudotime\n (p={p:.3f})')
ax.legend(labels=['Early Treg', 'Late Treg'], frameon=False, fontsize=9)
sns.despine()
plt.tight_layout()
plt.savefig('tfr_early_vs_late_kde.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for sample in samples_ab:
    sub = adata[adata.obs['SAHA_Sample'] == sample]
    coords_sub = sub.obsm['spatial']
    
    fibro_mask = sub.obs['cell_type_short'] == 'OGN+RSPO3+ Fib'
    early_mask = sub.obs['cell_type_treg'] == 'Treg_Early'
    late_mask  = sub.obs['cell_type_treg'] == 'Treg_Late'
    
    if fibro_mask.sum() == 0:
        continue
    
    tree = KDTree(coords_sub[fibro_mask])
    
    early_dists, _ = tree.query(coords_sub[early_mask])
    late_dists,  _ = tree.query(coords_sub[late_mask])
    
    # Convert to µm
    early_dist_um = early_dists * 0.18
    late_dist_um  = late_dists  * 0.18
    
    _, p = mannwhitneyu(late_dist_um, early_dist_um, alternative='two-sided')
    
    print(f"{sample}:")
    print(f"  Early Treg → nearest fibroblast: {np.median(early_dist_um):.1f}µm")
    print(f"  Late Treg  → nearest fibroblast: {np.median(late_dist_um):.1f}µm")
    print(f"  p = {p:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(5,3))

col_active = '#888780'
col_post   = '#4E9F8C'
col_early  = '#378ADD'
col_late   = '#D85A30'

x_labels = ['Active IBD', 'Post-treatment\nRemission']

# ── 1. OGN+RSPO3+ fibroblast proportion ──────────────────────────────────────
vals_ogn = [data['SAHA_post_trt_nonremission']['ogn_pct'], data['SAHA_pre_trt']['ogn_pct']]
bars = axes[0].bar([0, 1], vals_ogn,
                   color=[col_active, col_post],
                   width=0.5, edgecolor='none')
for bar, val in zip(bars, vals_ogn):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(x_labels, fontsize=8)
axes[0].set_ylabel('% of connective cells', fontsize=9)
axes[0].set_title('OGN⁺RSPO3⁺ fibroblast\nabundance', fontsize=10, pad=6)
axes[0].set_ylim(0, max(vals_ogn) * 1.25)
axes[0].set_xlim(-0.5, 1.5)
axes[0].tick_params(axis='both', labelsize=8)
sns.despine(ax=axes[0])

# ── 2. Early vs Late Treg stacked ────────────────────────────────────────────
for i, timepoint in enumerate(['SAHA_post_trt_nonremission', 'SAHA_pre_trt']):
    early_val = data[timepoint]['early_pct']
    late_val  = data[timepoint]['late_pct']

    axes[1].bar(i, early_val, width=0.5, color=col_early, edgecolor='none')
    axes[1].bar(i, late_val,  width=0.5, color=col_late,
                bottom=early_val, edgecolor='none')
    axes[1].text(i, early_val/2, f'{early_val:.0f}%',
                 ha='center', va='center', fontsize=10,
                 fontweight='bold', color='white')
    axes[1].text(i, early_val + late_val/2, f'{late_val:.0f}%',
                 ha='center', va='center', fontsize=10,
                 fontweight='bold', color='white')

axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(x_labels, fontsize=8)
axes[1].set_ylabel('Treg proportion (%)', fontsize=9)
axes[1].set_title('Early vs Late Treg\nproportion', fontsize=10, pad=6)
axes[1].set_xlim(-0.5, 1.5)
axes[1].set_ylim(0, 105)
axes[1].tick_params(axis='both', labelsize=8)
sns.despine(ax=axes[1])

# ── Shared legend ─────────────────────────────────────────────────────────────
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=col_active, label='Active IBD'),
    Patch(facecolor=col_post,   label='Post-trt Remission'),
    Patch(facecolor=col_early,  label='Early Treg'),
    Patch(facecolor=col_late,   label='Late Treg'),
]
fig.legend(handles=legend_elements, frameon=False, fontsize=8,
           loc='lower center', bbox_to_anchor=(0.5, -0.05),
           ncol=4)

plt.suptitle('',
             fontsize=10, y=1.02)
plt.tight_layout(w_pad=3)
plt.savefig('comparison_final.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4,3.5))

x = np.array([0, 0.8])  # two groups — OGN and ADAMDEC
width = 0.25

for i, (timepoint, color, label) in enumerate(zip(
    ['SAHA_post_trt_nonremission', 'SAHA_pre_trt'],
    [col_active, col_post],
    ['Active IBD', 'Post-trt Remission']
)):
    ogn_val  = data[timepoint]['ogn_pct']
    adam_val = data[timepoint]['adamdec_pct']

    ax.bar(x[0] + i*width - width/2, ogn_val,  width,
           color=color, label=label, edgecolor='none')
    ax.bar(x[1] + i*width - width/2, adam_val, width,
           color=color, edgecolor='none')

    ax.text(x[0] + i*width - width/2, ogn_val  + 0.2,
            f'{ogn_val:.1f}%', ha='center', fontsize=8, fontweight='bold')
    ax.text(x[1] + i*width - width/2, adam_val + 0.2,
            f'{adam_val:.1f}%', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(['OGN⁺RSPO3⁺', 'ADAMDEC1⁺'], fontsize=9)
ax.set_ylabel('% of Fibroblasts', fontsize=9)
ax.set_title('Fibroblast Abundance\nActive IBD vs Post-trt Remission', fontsize=10)
ax.set_xlim(-0.4, 1.2)
ax.set_ylim(0, max(data['SAHA_post_trt_nonremission']['ogn_pct'],
                   data['SAHA_post_trt_nonremission']['adamdec_pct']) * 1.3)
ax.legend(frameon=False, fontsize=8,
          loc='upper right')
sns.despine()
plt.tight_layout()
plt.savefig('fibro_subtypes.pdf', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
spatial=adata.copy()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

samples = {
    'SAHA_post_trt_nonremission': 'Pre-treatment',
    'SAHA_pre_trt': 'Post-treatment Remission', 
    'SAHA_post_trt_remission': 'Post-treatment Non-Remission'
}

for col, (sample_id, sample_label) in enumerate(samples.items()):
    
    sample_mask = adata.obs['SAHA_Sample'] == sample_id
    sample_adata = adata[sample_mask].copy()
    
    x = sample_adata.obs['x'].values
    y = sample_adata.obs['y'].values
    
    # Get cell type masks
    treg_mask = sample_adata.obs['cell_type_short'].str.contains(
        'Treg|treg', case=False, na=False
    )
    late_treg_mask = sample_adata.obs['cell_type_treg'].str.contains(
        'late|Late', case=False, na=False
    )
    early_treg_mask = sample_adata.obs['cell_type_treg'].str.contains(
        'early|Early', case=False, na=False
    )
    ogn_fib_mask = sample_adata.obs['cell_type'].str.contains(
        'OGN|RSPO3', case=False, na=False
    )
    
    # Get CCL20 and TNFRSF4 expression
    for gene in ['CCL20', 'TNFRSF4']:
        if gene in sample_adata.var_names:
            expr = sample_adata[:, gene].layers['normalized']
            if hasattr(expr, 'toarray'):
                expr = expr.toarray().flatten()
            else:
                expr = np.array(expr).flatten()
            sample_adata.obs[f'{gene}_expr'] = expr

    # CCL20 high Tregs
    if 'CCL20_expr' in sample_adata.obs.columns:
        ccl20_thresh = np.percentile(
            sample_adata.obs.loc[treg_mask, 'CCL20_expr'], 75
        ) if treg_mask.sum() > 0 else 0
        ccl20_high_treg = treg_mask & (
            sample_adata.obs['CCL20_expr'] > ccl20_thresh
        )
    else:
        ccl20_high_treg = pd.Series(False, index=sample_adata.obs.index)

    # TNFRSF4 high Tregs
    if 'TNFRSF4_expr' in sample_adata.obs.columns:
        tnfrsf4_thresh = np.percentile(
            sample_adata.obs.loc[treg_mask, 'TNFRSF4_expr'], 75
        ) if treg_mask.sum() > 0 else 0
        tnfrsf4_high_treg = treg_mask & (
            sample_adata.obs['TNFRSF4_expr'] > tnfrsf4_thresh
        )
    else:
        tnfrsf4_high_treg = pd.Series(False, index=sample_adata.obs.index)

    # ── Row 1: CCL20 spatial map ──────────────────────────────────────────
    ax = axes[0, col]

    # Background cells
    ax.scatter(
        x, y,
        c='#f0f0f0', s=1, alpha=0.3, rasterized=True, linewidths=0
    )
    # OGN+RSPO3+ fibroblasts
    ax.scatter(
        x[ogn_fib_mask], y[ogn_fib_mask],
        c='#2ecc71', s=8, alpha=0.7,
        label='OGN⁺RSPO3⁺ Fib', rasterized=True, linewidths=0
    )
    # All Tregs (low CCL20)
    non_ccl20_treg = treg_mask & ~ccl20_high_treg
    ax.scatter(
        x[non_ccl20_treg], y[non_ccl20_treg],
        c='#aec6e8', s=8, alpha=0.6,
        label='Treg (CCL20 low)', rasterized=True, linewidths=0
    )
    # CCL20-high Tregs
    ax.scatter(
        x[ccl20_high_treg], y[ccl20_high_treg],
        c='#D85A30', s=15, alpha=0.9,
        label='Treg (CCL20 high)', rasterized=True, linewidths=0,
        zorder=5
    )

    ax.set_title(
        f'{sample_label}\nCCL20⁺ Tregs & OGN⁺RSPO3⁺ Fibroblasts',
        fontsize=10, fontweight='bold'
    )
    ax.set_xlabel('X (px)', fontsize=8)
    ax.set_ylabel('Y (px)', fontsize=8)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=7)

    # Cell counts annotation
    ax.text(
        0.02, 0.98,
        f'CCL20⁺ Tregs: {ccl20_high_treg.sum()}\n'
        f'OGN⁺RSPO3⁺ Fib: {ogn_fib_mask.sum()}',
        transform=ax.transAxes,
        fontsize=7, va='top', ha='left',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7)
    )

    # ── Row 2: TNFRSF4 spatial map ───────────────────────────────────────
    ax = axes[1, col]

    # Background cells
    ax.scatter(
        x, y,
        c='#f0f0f0', s=1, alpha=0.3, rasterized=True, linewidths=0
    )
    # OGN+RSPO3+ fibroblasts
    ax.scatter(
        x[ogn_fib_mask], y[ogn_fib_mask],
        c='#2ecc71', s=8, alpha=0.7,
        label='OGN⁺RSPO3⁺ Fib', rasterized=True, linewidths=0
    )
    # All Tregs (low TNFRSF4)
    non_tnfrsf4_treg = treg_mask & ~tnfrsf4_high_treg
    ax.scatter(
        x[non_tnfrsf4_treg], y[non_tnfrsf4_treg],
        c='#aec6e8', s=8, alpha=0.6,
        label='Treg (TNFRSF4 low)', rasterized=True, linewidths=0
    )
    # TNFRSF4-high Tregs
    ax.scatter(
        x[tnfrsf4_high_treg], y[tnfrsf4_high_treg],
        c='#8B0000', s=15, alpha=0.9,
        label='Treg (TNFRSF4 high)', rasterized=True, linewidths=0,
        zorder=5
    )

    ax.set_title(
        f'{sample_label}\nTNFRSF4⁺ Tregs & OGN⁺RSPO3⁺ Fibroblasts',
        fontsize=10, fontweight='bold'
    )
    ax.set_xlabel('X (px)', fontsize=8)
    ax.set_ylabel('Y (px)', fontsize=8)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=7)

    ax.text(
        0.02, 0.98,
        f'TNFRSF4⁺ Tregs: {tnfrsf4_high_treg.sum()}\n'
        f'OGN⁺RSPO3⁺ Fib: {ogn_fib_mask.sum()}',
        transform=ax.transAxes,
        fontsize=7, va='top', ha='left',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7)
    )

# Shared legends
legend_elements_row1 = [
    mpatches.Patch(color='#f0f0f0', label='Other cells'),
    mpatches.Patch(color='#2ecc71', label='OGN⁺RSPO3⁺ Fib'),
    mpatches.Patch(color='#aec6e8', label='Treg (CCL20 low)'),
    mpatches.Patch(color='#D85A30', label='Treg (CCL20 high)'),
]
legend_elements_row2 = [
    mpatches.Patch(color='#f0f0f0', label='Other cells'),
    mpatches.Patch(color='#2ecc71', label='OGN⁺RSPO3⁺ Fib'),
    mpatches.Patch(color='#aec6e8', label='Treg (TNFRSF4 low)'),
    mpatches.Patch(color='#8B0000', label='Treg (TNFRSF4 high)'),
]

axes[0, 2].legend(
    handles=legend_elements_row1,
    loc='lower right', fontsize=7,
    framealpha=0.8, bbox_to_anchor=(1.0, 0.0)
)
axes[1, 2].legend(
    handles=legend_elements_row2,
    loc='lower right', fontsize=7,
    framealpha=0.8, bbox_to_anchor=(1.0, 0.0)
)

plt.suptitle(
    'Spatial Distribution of Resistance-Persistent Treg Populations\n'
    'and OGN⁺RSPO3⁺ Fibroblasts Across Treatment Response Groups',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(
    'spatial_resistance_maps.pdf',
    dpi=300, bbox_inches='tight'
)
plt.show()

In [ ]:
# Extract spatial coordinates from obsm
adata.obs['x'] = adata.obsm['spatial'][:, 0]
adata.obs['y'] = adata.obsm['spatial'][:, 1]

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import KDTree
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns

# Make sure coordinates are extracted
adata.obs['x'] = adata.obsm['spatial'][:, 0]
adata.obs['y'] = adata.obsm['spatial'][:, 1]

# Extract normalized expression for CCL20 and TNFRSF4
for gene in ['CCL20', 'TNFRSF4']:
    if gene in adata.var_names:
        expr = adata[:, gene].layers['normalized']
        if hasattr(expr, 'toarray'):
            expr = expr.toarray().flatten()
        else:
            expr = np.array(expr).flatten()
        adata.obs[f'{gene}_expr'] = expr


In [ ]:
# Fix FOV filter first
def fov_coloc_score(adata, gene, radius_px=278):
    results = []
    for sample_id in adata.obs['SAHA_Sample'].unique():
        sample = adata[adata.obs['SAHA_Sample'] == sample_id].copy()
        for fov in sample.obs['fov'].unique():
            fov_data = sample[sample.obs['fov'] == fov].copy()
            if len(fov_data) < 10:
                continue
            treg_mask = fov_data.obs['cell_type_short'].str.contains(
                'Treg|treg', case=False, na=False
            )
            ogn_mask = fov_data.obs['cell_type'].str.contains(
                'OGN|RSPO3', case=False, na=False
            )
            # Relaxed filter
            if treg_mask.sum() < 1 or ogn_mask.sum() < 1:
                continue
            treg_expr = fov_data.obs.loc[treg_mask, f'{gene}_expr']
            thresh = np.percentile(treg_expr, 75)
            gene_high_treg = treg_mask & (fov_data.obs[f'{gene}_expr'] > thresh)
            if gene_high_treg.sum() == 0:
                continue
            coords = np.column_stack([
                fov_data.obs['x'].values,
                fov_data.obs['y'].values
            ])
            ogn_coords = coords[ogn_mask.values]
            treg_high_coords = coords[gene_high_treg.values]
            tree = KDTree(ogn_coords)
            distances, _ = tree.query(treg_high_coords, k=1)
            frac_proximal = np.mean(distances <= radius_px)
            results.append({
                'SAHA_Sample': sample_id,
                'fov': fov,
                'frac_proximal': frac_proximal,
                'n_gene_high_treg': gene_high_treg.sum(),
                'n_ogn_fib': ogn_mask.sum(),
                'gene': gene
            })
    return pd.DataFrame(results)

# Cell-level proximity
def cell_level_proximity(adata, gene, radius_px=278):
    results = []
    for sample_id in adata.obs['SAHA_Sample'].unique():
        sample = adata[adata.obs['SAHA_Sample'] == sample_id].copy()
        treg_mask = sample.obs['cell_type_short'].str.contains(
            'Treg|treg', case=False, na=False
        )
        ogn_mask = sample.obs['cell_type'].str.contains(
            'OGN|RSPO3', case=False, na=False
        )
        if treg_mask.sum() < 1 or ogn_mask.sum() < 1:
            continue
        coords = np.column_stack([
            sample.obs['x'].values,
            sample.obs['y'].values
        ])
        ogn_coords = coords[ogn_mask.values]
        treg_coords = coords[treg_mask.values]
        treg_expr = sample.obs.loc[treg_mask, f'{gene}_expr'].values
        tree = KDTree(ogn_coords)
        distances, _ = tree.query(treg_coords, k=1)
        thresh = np.percentile(treg_expr, 75)
        for dist, expr in zip(distances, treg_expr):
            results.append({
                'SAHA_Sample': sample_id,
                'distance_to_ogn_px': dist,
                'distance_to_ogn_um': dist * 0.18,
                'gene_expr': expr,
                'gene_high': expr > thresh,
                'gene': gene
            })
    return pd.DataFrame(results)

# Run both
coloc_ccl20 = fov_coloc_score(adata, 'CCL20', radius_px=278)
coloc_tnfrsf4 = fov_coloc_score(adata, 'TNFRSF4', radius_px=278)
coloc_df = pd.concat([coloc_ccl20, coloc_tnfrsf4])

cell_ccl20 = cell_level_proximity(adata, 'CCL20')
cell_tnfrsf4 = cell_level_proximity(adata, 'TNFRSF4')

print("=== FOV-level co-localization ===")
print(coloc_df.groupby(['SAHA_Sample', 'gene'])['frac_proximal'].describe())

print("\n=== Cell-level: gene-high vs gene-low distance to OGN+RSPO3+ Fib ===")
for gene, df in [('CCL20', cell_ccl20), ('TNFRSF4', cell_tnfrsf4)]:
    print(f"\n{gene}:")
    for sample_id in ['SAHA_post_trt_nonremission', 'SAHA_pre_trt', 'SAHA_post_trt_remission']:
        s = df[df['SAHA_Sample'] == sample_id]
        if len(s) == 0:
            continue
        high = s[s['gene_high']]['distance_to_ogn_um'].values
        low = s[~s['gene_high']]['distance_to_ogn_um'].values
        _, p = mannwhitneyu(high, low, alternative='less')
        print(f"  {sample_id}: "
              f"high={high.mean():.1f}um, "
              f"low={low.mean():.1f}um, "
              f"p={p:.4f} "
              f"(n_high={len(high)}, n_low={len(low)})")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import matplotlib.colors as mcolors

# Compute Th17 score on full adata
adata_tmp = adata.copy()
adata_tmp.X = adata_tmp.layers['normalized']

th17_genes = ['RORC', 'CCR6', 'IL17A', 'IL17F']
th17_genes_present = [g for g in th17_genes if g in adata_tmp.var_names]
print(f"Th17 genes in panel: {th17_genes_present}")

sc.tl.score_genes(adata_tmp, th17_genes_present, score_name='th17_score')
adata.obs['th17_score'] = adata_tmp.obs['th17_score']

# Extract CCL20 expression
ccl20_expr = adata_tmp[:, 'CCL20'].X
if hasattr(ccl20_expr, 'toarray'):
    ccl20_expr = ccl20_expr.toarray().flatten()
else:
    ccl20_expr = np.array(ccl20_expr).flatten()
adata.obs['CCL20_expr'] = ccl20_expr

# Extract coordinates
adata.obs['x'] = adata.obsm['spatial'][:, 0]
adata.obs['y'] = adata.obsm['spatial'][:, 1]

# Setup
samples = {
    'SAHA_post_trt_nonremission': 'Pre-treatment',
    'SAHA_pre_trt': 'Post-trt Remission',
    'SAHA_post_trt_remission': 'Post-trt Non-Remission'
}

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for col, (sample_id, sample_label) in enumerate(samples.items()):

    sample_mask = adata.obs['SAHA_Sample'] == sample_id
    sample_adata = adata[sample_mask].copy()

    x = sample_adata.obs['x'].values
    y = sample_adata.obs['y'].values

    # Cell type masks
    treg_mask = sample_adata.obs['cell_type_short'].str.contains(
        'Treg|treg', case=False, na=False
    ).values
    ogn_mask = sample_adata.obs['cell_type'].str.contains(
        'OGN|RSPO3', case=False, na=False
    ).values

    treg_x = x[treg_mask]
    treg_y = y[treg_mask]
    ogn_x = x[ogn_mask]
    ogn_y = y[ogn_mask]

    ccl20_vals = sample_adata.obs['CCL20_expr'].values[treg_mask]
    th17_vals = sample_adata.obs['th17_score'].values[treg_mask]

    # Shared color scale across samples for comparability
    ccl20_vmin = adata.obs.loc[
        adata.obs['cell_type_short'].str.contains('Treg|treg', case=False, na=False),
        'CCL20_expr'
    ].quantile(0.05)
    ccl20_vmax = adata.obs.loc[
        adata.obs['cell_type_short'].str.contains('Treg|treg', case=False, na=False),
        'CCL20_expr'
    ].quantile(0.95)

    th17_vmin = adata.obs.loc[
        adata.obs['cell_type_short'].str.contains('Treg|treg', case=False, na=False),
        'th17_score'
    ].quantile(0.05)
    th17_vmax = adata.obs.loc[
        adata.obs['cell_type_short'].str.contains('Treg|treg', case=False, na=False),
        'th17_score'
    ].quantile(0.95)

    for row, (vals, vmin, vmax, cmap, score_name) in enumerate(zip(
        [ccl20_vals, th17_vals],
        [ccl20_vmin, th17_vmin],
        [ccl20_vmax, th17_vmax],
        ['YlOrRd', 'PuRd'],
        ['CCL20 Expression', 'Th17 Score\n(RORC, CCR6, IL17A, IL17F)']
    )):
        ax = axes[row, col]

        # Background cells — light grey
        ax.scatter(
            x, y,
            c='#f0f0f0', s=1, alpha=0.2,
            rasterized=True, linewidths=0
        )

        # OGN+RSPO3+ fibroblasts — green
        ax.scatter(
            ogn_x, ogn_y,
            c='#2ecc71', s=6, alpha=0.6,
            rasterized=True, linewidths=0,
            zorder=2
        )

        # Tregs colored by score
        sc_plot = ax.scatter(
            treg_x, treg_y,
            c=vals,
            cmap=cmap,
            vmin=vmin, vmax=vmax,
            s=18, alpha=0.9,
            rasterized=True, linewidths=0,
            zorder=3
        )

        # Colorbar
        cbar = plt.colorbar(sc_plot, ax=ax, shrink=0.6, pad=0.02)
        cbar.set_label(score_name.split('\n')[0], fontsize=7)
        cbar.ax.tick_params(labelsize=6)

        # Title
        if row == 0:
            ax.set_title(
                f'{sample_label}\n{score_name}',
                fontsize=9, fontweight='bold'
            )
        else:
            ax.set_title(score_name, fontsize=9)

        # Cell count annotation
        ax.text(
            0.02, 0.98,
            f'Tregs: {treg_mask.sum()}\nOGN⁺RSPO3⁺: {ogn_mask.sum()}',
            transform=ax.transAxes,
            fontsize=7, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
        )

        ax.set_aspect('equal')
        ax.axis('off')

# Row labels
axes[0, 0].set_ylabel('CCL20 Expression\nin Tregs', fontsize=10, labelpad=10)
axes[1, 0].set_ylabel('Th17 Score\nin Tregs', fontsize=10, labelpad=10)
for ax in axes[:, 0]:
    ax.axis('on')
    ax.set_yticks([])
    ax.set_xticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#f0f0f0',
           markersize=8, label='Other cells'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71',
           markersize=8, label='OGN⁺RSPO3⁺ Fibroblasts'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#D85A30',
           markersize=8, label='Tregs (colored by score)'),
]
fig.legend(
    handles=legend_elements,
    loc='lower center', ncol=3,
    fontsize=9, frameon=False,
    bbox_to_anchor=(0.5, -0.01)
)

fig.suptitle(
    'CCL20 Expression and Th17 Gene Score in Tregs\n'
    'Across Treatment Response Groups',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('spatial_ccl20_th17_tregs.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Print summary stats
print("\n=== CCL20 and Th17 score in Tregs per sample ===")
treg_mask_full = adata.obs['cell_type_short'].str.contains(
    'Treg|treg', case=False, na=False
)
print(adata.obs[treg_mask_full].groupby('SAHA_Sample')[
    ['CCL20_expr', 'th17_score']
].describe())

In [ ]:
# First check Th17 score distribution across cell types
print("Th17 score by cell type:")
print(adata.obs.groupby('cell_type_short')['th17_score'].describe().sort_values(
    'mean', ascending=False
).head(15))

In [ ]:

sample_labels = {
    'SAHA_post_trt_nonremission': 'Pre-trt',
    'SAHA_pre_trt': 'Post-trt Remission',
    'SAHA_post_trt_remission': 'Post-trt Non-Remission'
}
palette_samples = {
    'SAHA_post_trt_nonremission': '#888888',
    'SAHA_pre_trt': '#378ADD',
    'SAHA_post_trt_remission': '#D85A30'
}


In [ ]:
# How many Tregs express either CCL20 or TNFRSF4
for sample_id in ['SAHA_post_trt_nonremission', 'SAHA_pre_trt', 'SAHA_post_trt_remission']:
    sample_mask = adata.obs['SAHA_Sample'] == sample_id
    treg_mask = adata.obs['cell_type_short'].str.contains(
        'Treg|treg', case=False, na=False
    )
    mask = treg_mask & sample_mask
    
    ccl20 = adata.obs.loc[mask, 'CCL20_expr'] > 0
    tnfrsf4 = adata.obs.loc[mask, 'TNFRSF4_expr'] > 0
    either = ccl20 | tnfrsf4
    both = ccl20 & tnfrsf4
    
    print(f"\n{sample_id} (n Tregs={mask.sum()}):")
    print(f"  CCL20+: {ccl20.sum()}")
    print(f"  TNFRSF4+: {tnfrsf4.sum()}")
    print(f"  Either: {either.sum()}")
    print(f"  Both: {both.sum()}")

In [ ]:
# Make sure resistance_score is in adata.obs
if 'resistance_score' not in adata.obs.columns:
    adata_tmp = adata.copy()
    adata_tmp.X = adata_tmp.layers['normalized']
    sc.tl.score_genes(
        adata_tmp,
        ['CCL20', 'TNFRSF4'],
        score_name='resistance_score'
    )
    adata.obs['resistance_score'] = adata_tmp.obs['resistance_score']
    print("resistance_score added")

# Also make sure CCL20_expr and TNFRSF4_expr are there
for gene in ['CCL20', 'TNFRSF4']:
    if f'{gene}_expr' not in adata.obs.columns:
        expr = adata_tmp[:, gene].X
        if hasattr(expr, 'toarray'):
            expr = expr.toarray().flatten()
        else:
            expr = np.array(expr).flatten()
        adata.obs[f'{gene}_expr'] = expr
        print(f"{gene}_expr added")

# Verify
print(adata.obs[['resistance_score', 'CCL20_expr', 'TNFRSF4_expr']].describe())

In [ ]:
def get_fib_masks(sample_adata):
    return {
        'OGN+RSPO3+':         (sample_adata.obs['cell_type'] == 'OGN+RSPO3+ Fib').values,
        'T cell-Interacting': (sample_adata.obs['cell_type'] == 'T cell-Interacting Fib').values,
        'Activated':          (sample_adata.obs['cell_type'] == 'Activated Fib').values,
        'Myofibroblast':      (sample_adata.obs['cell_type'].str.contains('Myofibroblast', case=False, na=False)).values,
        'ADAMDEC1+':          (sample_adata.obs['cell_type'] == 'ADAMDEC1+ Fib').values,
    }

fib_colors = {
    'OGN+RSPO3+':         '#2ecc71',
    'T cell-Interacting': '#9b59b6',
    'Activated':          '#e74c3c',
    'Myofibroblast':      '#E59F07',
    'ADAMDEC1+':          '#3498db',
}

fib_masks = get_fib_masks(adata)

In [ ]:
# Rerun concat
all_results = pd.concat([
    neighbor_scores_multifib(adata, s)
    for s in ['SAHA_post_trt_nonremission', 'SAHA_pre_trt', 'SAHA_post_trt_remission']
], ignore_index=True)

print(type(all_results))  # should be <class 'pandas.core.frame.DataFrame'>
print(all_results.shape)
print(all_results['SAHA_Sample'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11,3.5), sharey=True)


for ax, sample_id in zip(
    axes, ['SAHA_post_trt_nonremission', 'SAHA_pre_trt', 'SAHA_post_trt_remission']
):
    s = all_results[all_results['SAHA_Sample'] == sample_id]
    means_near = []
    sems_near = []
    means_far = []
    sems_far = []
    p_vals = []
    valid_names = []

    for name in fib_masks.keys():
        col = f'near_{name}'
        if col not in s.columns:
            continue
        near = s[s[col]]['resistance_score'].values
        far = s[~s[col]]['resistance_score'].values
        if len(near) < 2 or len(far) < 2:
            continue
        #_, p = mannwhitneyu(near, far, alternative='greater')
        _, p = mannwhitneyu(near, far, alternative='two-sided')

        means_near.append(np.mean(near))
        sems_near.append(np.std(near) / np.sqrt(len(near)))
        means_far.append(np.mean(far))
        sems_far.append(np.std(far) / np.sqrt(len(far)))
        p_vals.append(p)
        valid_names.append(name)

    x = np.arange(len(valid_names))
    width = 0.38

    # Near bars — hatched
    ax.bar(
        x - width/2, means_near, width,
        color=[fib_colors[n] for n in valid_names],
        alpha=0.85, hatch='///',
        edgecolor='white', linewidth=0.5
    )
    # Far bars — solid
    ax.bar(
        x + width/2, means_far, width,
        color=[fib_colors[n] for n in valid_names],
        alpha=0.4, edgecolor='white', linewidth=0.5
    )
    # P-value brackets
    for i, (p, mn, mf, sem_n, sem_f) in enumerate(
        zip(p_vals, means_near, means_far, sems_near, sems_far)
    ):
        top = min(max(mn, mf) + 0.01, 0.65)
        p_str = f'p={p:.3f}' if p >= 0.001 else 'p<0.001'
        color_p = '#111111' if p < 0.05 else '#5C5B5B'
        weight = 'bold' if p < 0.05 else 'normal'
        ax.plot(
            [i - width/2, i - width/2, i + width/2, i + width/2],
            [top, top + 0.01, top + 0.01, top],
            color=color_p, lw=0.8
        )
        ax.text(
            i, top + 0.015,
            p_str,
            ha='center', va='bottom',
            fontsize=7, color=color_p,
            fontweight=weight
        )

    # X axis labels
    ax.set_xticks(x)
    ax.set_xticklabels(
        [n.replace('T cell-Interacting', 'T cell-\nInteracting')
          .replace('Myofibroblast', 'Myofib')
         for n in valid_names],
        fontsize=7.5, ha='center'
    )

    # Y axis label only on first panel
    if ax == axes[0]:
        ax.set_ylabel('Mean Resistance Score \n(CCL20+TNFRSF4)', fontsize=9)
    else:
        ax.set_ylabel('')

    # Panel title with treatment label and n
    n_tregs = len(s)
    n_near_ogn = int(s[s['near_OGN+RSPO3+']].shape[0]) \
        if 'near_OGN+RSPO3+' in s.columns else 0
    
    ax.set_title(
        f'{sample_labels[sample_id]}\n'
        f'n={n_tregs} Tregs',
        fontsize=10, pad=6
    )
    ax.set_ylim(-0.05, 0.6)
    ax.axhline(0, color='grey', lw=0.5, linestyle='--', alpha=0.7)
    ax.tick_params(axis='both', labelsize=8)
    sns.despine(ax=ax)

# Bottom legend — one row
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Near/far pattern legend

legend_elements = [
    Patch(facecolor='#555555', alpha=0.85, hatch='///',
          edgecolor='white', linewidth=0.5,
          label='Near fibroblast (≤50µm)'),
    Patch(facecolor='#555555', alpha=0.4,
          edgecolor='white', linewidth=0.5,
          label='Far from fibroblast (>50µm)'),
]

fig.legend(
    handles=legend_elements,
    loc='upper center',
    ncol=1,
    fontsize=9,
    frameon=False,
    bbox_to_anchor=(0.53, 0.88),
    handlelength=1.5,
    handleheight=1.0,
    columnspacing=1.0
)
fig.text(
    0.5, 0.03,
    'Fibroblasts',
    ha='center', va='center',
    fontsize=10, 
)
plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.savefig(
    'resistance_score_multifib_publication.pdf',
    dpi=300, bbox_inches='tight'
)
plt.show()